# Lab 16-07 · From Analog Prototypes to Digital Biquads

Five parts, each built on `iirdesign.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import iirdesign as iir
sys.path.insert(0, "../16-06-lab")
import firdesign as fd

fs = 48000.0
wp, ws = 2 * np.pi * 4000 / fs, 2 * np.pi * 5000 / fs

## Part 1 · Warping and prewarping

*Predict:* will the unprewarped 1 kHz filter's −3 dB point be above or below 1 kHz?

In [2]:
fa = np.linspace(0, 60e3, 601)
plt.figure(figsize=(5, 3.2)); plt.plot(fa / 1e3, [iir.unwarp(2 * np.pi * f, fs) / 1e3 for f in fa]); plt.plot([0, 24], [0, 24], "--")
plt.xlabel("analog kHz"); plt.ylabel("digital kHz"); plt.show()
for pre in (False, True):
    b, a = iir.bilinear_first_order(1000, 8000, prewarped=pre)
    w = np.linspace(0, np.pi, 400001)
    _, H = signal.freqz(b, a, worN=w)
    f3 = w[np.argmin(np.abs(np.abs(H) - 1 / math.sqrt(2)))] * 8000 / (2 * np.pi)
    print(f"prewarped={pre}: b = {np.round(b, 4)}, a = {np.round(a, 4)}, -3 dB at {f3:.1f} Hz")
    assert abs(f3 - (1000.0 if pre else 952.9)) < 0.1

prewarped=False: b = [0.282 0.282], a = [ 1.     -0.4361], -3 dB at 952.9 Hz
prewarped=True: b = [0.2929 0.2929], a = [ 1.     -0.4142], -3 dB at 1000.0 Hz


## Part 2 · Impulse invariance aliases

*Predict:* at which cutoff does the impulse-invariant design's Nyquist/DC ratio first exceed the analog ratio by 0.1?

In [3]:
fcs = np.linspace(250, 3000, 12)
dig, ana = [], []
for fc in fcs:
    b, a = iir.impulse_invariant_first_order(fc, 8000)
    _, H = signal.freqz(b, a, worN=[0, np.pi])
    dig.append(abs(H[1]) / abs(H[0]))
    ana.append(1 / math.sqrt(1 + (4000 / fc) ** 2))
dig, ana = np.array(dig), np.array(ana)
for fc, d, an in zip(fcs, dig, ana):
    print(f"fc = {fc:6.0f} Hz: digital {d:.3f}, analog {an:.3f}")
assert np.all(dig > ana)
b, a = iir.impulse_invariant_first_order(1000, 8000)
_, H = signal.freqz(b, a, worN=[0, np.pi])
assert abs(abs(H[1]) / abs(H[0]) - 0.374) < 0.0005 and abs(abs(H[0]) - 1.444) < 0.0005
plt.figure(figsize=(5, 3)); plt.plot(fcs, dig, label="impulse invariant"); plt.plot(fcs, ana, label="analog at 4 kHz")
plt.xlabel("cutoff (Hz)"); plt.ylabel("gain at Nyquist / DC"); plt.legend(); plt.show()

fc =    250 Hz: digital 0.098, analog 0.062
fc =    500 Hz: digital 0.194, analog 0.124
fc =    750 Hz: digital 0.286, analog 0.184
fc =   1000 Hz: digital 0.374, analog 0.243
fc =   1250 Hz: digital 0.455, analog 0.298
fc =   1500 Hz: digital 0.529, analog 0.351
fc =   1750 Hz: digital 0.596, analog 0.401
fc =   2000 Hz: digital 0.656, analog 0.447
fc =   2250 Hz: digital 0.708, analog 0.490
fc =   2500 Hz: digital 0.754, analog 0.530
fc =   2750 Hz: digital 0.793, analog 0.567
fc =   3000 Hz: digital 0.827, analog 0.600


## Part 3 · Prototypes from their poles

*Predict:* will the order-38 Butterworth filter meet 60 dB with margin or only just?

In [4]:
w = np.linspace(1e-5, np.pi, 20001)
def check(z, p, k):
    """Passband ripple (dB) and stopband attenuation (dB); these prototypes fall monotonically in the stopband,
    so its worst point is the edge itself, which a grid can miss: evaluate there exactly and confirm."""
    m = np.abs(iir.response(z, p, k, w))
    edge = abs(iir.response(z, p, k, [ws])[0])
    assert m[w >= ws].max() <= edge * (1 + 1e-9)
    return 20 * math.log10(m[w <= wp].max() / m[w <= wp].min()), -20 * math.log10(edge)
designs = {}
for name, N, fn in (("Chebyshev I", 14, iir.cheb1_analog), ("Butterworth", 38, iir.butter_analog)):
    z, p, k = iir.bilinear(*fn(N, 0.1, iir.prewarp(4000, fs)), fs)
    designs[name] = (z, p, k)
    rip, att = check(z, p, k)
    print(f"{name}, order {N}: ripple {rip:.4f} dB, attenuation {att:.2f} dB, largest pole radius {abs(p).max():.5f}")
zc, pc, kc = designs["Chebyshev I"]
_, Hs = signal.sosfreqz(signal.cheby1(14, 0.1, 4000, fs=fs, output="sos"), worN=w)
assert np.max(np.abs(iir.response(zc, pc, kc, w) - Hs)) < 1e-9
assert abs(check(zc, pc, kc)[1] - 64.62) < 0.005 and abs(check(*designs["Butterworth"])[1] - 61.75) < 0.005

Chebyshev I, order 14: ripple 0.1000 dB, attenuation 64.62 dB, largest pole radius 0.98970
Butterworth, order 38: ripple 0.0993 dB, attenuation 61.75 dB, largest pole radius 0.97867


## Part 4 · Three IIR filters and an FIR filter

*Predict:* which IIR filter has the largest passband group-delay spread?

In [5]:
sos_e = signal.ellip(8, 0.1, 60, 4000, fs=fs, output="sos")
ze, pe, ke = signal.sos2zpk(sos_e)
designs["elliptic"] = (ze, pe, ke)
wpb = np.linspace(1e-5, wp, 4001)   # the passband, including its edge, where the delay peaks
table = {}
for name, (z, p, k) in designs.items():
    gd = iir.group_delay(z, p, wpb)
    mults = iir.mults_sos(signal.zpk2sos(z, p, k)) if name != "elliptic" else iir.mults_sos(sos_e)
    table[name] = (mults, gd.max() - gd.min())
    print(f"{name:12s}: {mults:2d} multiplications/output, passband delay {gd.min():6.2f} to {gd.max():6.2f} samples")
dp, ds = fd.deltas(0.1, 60)
hf = fd.remez_lowpass(137, wp, ws, dp, ds)
print("FIR equiripple: 69 multiplications/output (folded), constant delay 68 samples")
assert table["elliptic"][0] == 14 and table["Chebyshev I"][0] == 17 and table["Butterworth"][0] == 41
assert abs(table["elliptic"][1] - 61.5) < 0.05 and abs(table["Chebyshev I"][1] - 89.4) < 0.05 and abs(table["Butterworth"][1] - 49.9) < 0.05
assert max(table, key=lambda n: table[n][1]) == "Chebyshev I"

Chebyshev I : 17 multiplications/output, passband delay  22.79 to 112.14 samples
Butterworth : 41 multiplications/output, passband delay  42.98 to  92.89 samples
elliptic    : 14 multiplications/output, passband delay   7.99 to  69.51 samples
FIR equiripple: 69 multiplications/output (folded), constant delay 68 samples


## Part 5 · One polynomial or four biquads

*Predict:* at how many fractional bits does the single polynomial first go unstable as you round more coarsely?

In [6]:
b, a = signal.sos2tf(sos_e)
print("direct-form denominator:", np.round(a, 3))
rows = []
for f in range(17, 7, -1):
    q = lambda x: np.round(x * 2.0 ** f) / 2.0 ** f
    r_direct = np.max(np.abs(np.roots(q(a))))
    r_sos = max(np.max(np.abs(np.roots(q(sec[3:])))) for sec in sos_e)
    rows.append((f, r_direct, r_sos))
    print(f"{f:2d} fractional bits: direct form {r_direct:.5f}{'  UNSTABLE' if r_direct >= 1 else ''}, sections {r_sos:.5f}")
first_unstable = next(f for f, rd, rs in rows if rd >= 1)
print("first unstable going down:", first_unstable, "bits")
assert first_unstable == 13 and all(rs < 1 for f, rd, rs in rows)
assert abs(dict((f, rd) for f, rd, rs in rows)[12] - 1.03017) < 5e-6

direct-form denominator: [  1.     -6.543  19.31  -33.469  37.187 -27.09   12.624  -3.439   0.419]
17 fractional bits: direct form 0.97834, sections 0.98499
16 fractional bits: direct form 0.98451, sections 0.98499
15 fractional bits: direct form 0.99407, sections 0.98498
14 fractional bits: direct form 0.97989, sections 0.98499
13 fractional bits: direct form 1.03223  UNSTABLE, sections 0.98499
12 fractional bits: direct form 1.03017  UNSTABLE, sections 0.98499
11 fractional bits: direct form 1.03544  UNSTABLE, sections 0.98499
10 fractional bits: direct form 1.10277  UNSTABLE, sections 0.98475
 9 fractional bits: direct form 1.09017  UNSTABLE, sections 0.98524
 8 fractional bits: direct form 1.13469  UNSTABLE, sections 0.98425
first unstable going down: 13 bits
